# 09 — Category & Sub-Category Trends Over Time
How sales mix shifts across product categories year over year.


In [ ]:
from pyhive import hive
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

# ── Global style ──────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("muted")

def get_conn():
    return hive.connect(host="hive-server2", port=10000,
                        database="default", auth="NONE")

def fetch_df(cur, sql):
    cur.execute(sql)
    cols = [d[0].split(".")[-1] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

def fmt_usd(v):
    if abs(v) >= 1_000_000:
        return f"${v/1_000_000:.2f}M"
    if abs(v) >= 1_000:
        return f"${v/1_000:.1f}K"
    return f"${v:.0f}"

conn = get_conn()
cur  = conn.cursor()
print("Connected.")


In [ ]:
# ── Annual Revenue by Category ────────────────────────────────────────────────
cat_year = fetch_df(cur, """
    SELECT SUBSTR(o.order_date,1,4) AS year,
           p.category,
           ROUND(SUM(oi.sales),2) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY SUBSTR(o.order_date,1,4), p.category
    ORDER BY year, category
""")

pivot = cat_year.pivot(index="year", columns="category",
                       values="revenue").fillna(0)

categories = pivot.columns.tolist()
years      = pivot.index.tolist()
bar_w      = 0.25
x          = range(len(years))

fig, ax = plt.subplots(figsize=(10, 5))
for k, (cat, col) in enumerate(zip(categories, PALETTE[:3])):
    offset = (k - 1) * bar_w
    bars   = ax.bar([i + offset for i in x], pivot[cat],
                    bar_w, color=col, label=cat, edgecolor="white", zorder=3)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 500,
                fmt_usd(bar.get_height()),
                ha="center", va="bottom", fontsize=7.5, color=col)

ax.set_xticks(list(x))
ax.set_xticklabels(years, fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
ax.legend(frameon=False, title="Category")
ax.set_title("Annual Revenue by Product Category")
ax.set_ylabel("Revenue (USD)")
ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
plt.tight_layout()
plt.show()


In [ ]:
# ── Sub-Category Revenue Heatmap ─────────────────────────────────────────────
sub_year = fetch_df(cur, """
    SELECT SUBSTR(o.order_date,1,4) AS year,
           p.sub_category,
           ROUND(SUM(oi.sales),2) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY SUBSTR(o.order_date,1,4), p.sub_category
    ORDER BY year, sub_category
""")

heat = sub_year.pivot(index="sub_category", columns="year",
                      values="revenue").fillna(0)

fig, ax = plt.subplots(figsize=(9, 9))
im = ax.imshow(heat.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, fontsize=11)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index, fontsize=9)

for i in range(len(heat.index)):
    for j in range(len(heat.columns)):
        val = heat.values[i, j]
        ax.text(j, i, fmt_usd(val), ha="center", va="center",
                fontsize=8, fontweight="bold",
                color="white" if val > heat.values.max()*0.6 else "black")

plt.colorbar(im, ax=ax, label="Revenue (USD)", shrink=0.6)
ax.set_title("Sub-Category Revenue Heatmap by Year", pad=12)
plt.tight_layout()
plt.show()


In [ ]:
cur.close()
conn.close()
print("Done.")
